# Suite XF — Transforms: models, procedures, application and certificates

A transform has a prototype, a procedure and a certificate ([CAP-074: transforms applied step by step](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-074.md)). Its
model is mbse-schemas data, installed by `Transforms.install` and defined by `Transforms.define`: its name, its
prototype (a pattern, which declares its parameters through `Parametrics`), the pattern of each frame it takes
(`source`, `sync`, `target`), the name its procedure is registered under, and what an application verifies. A
procedure is a native function registered by name, `procedure(source, sync, target, arguments)`, each frame a
`Visitors.OfPattern`. `Transforms.apply` applies a transform to frames already bound, each a `Transforms.Bound` of a
store and a frame, and writes the certificates ([CAP-075: steps linked to what they matched and wrote](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-075.md)): a history certificate,
the sequence of transform certificates and who decided each; a transform certificate per application, with its
parameters' values; and a sub-certificate per frame, its pattern instance and the transform before it on the same
history. The certificates live in a store of their own, the one the transforms are defined in.

The fixture is a shelf: a singleton `shelf.Shelf` owning `shelf.Item`s by key, each with a `name` and a `label`; the
pattern `shelf.Unlabelled`, an item without a label, which declares the parameter `case`; and `shelf.Whole`, the shelf.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Parametrics, Plain, Proxies, References, Transactions
from mbse.Transforms import Transforms
from support import raises


def shelf(txn):
    """shelf.Shelf, a singleton owning shelf.Items by key (shelf.Holds), each with a name and a label."""
    S = txn.Schemas
    return txn.uid("xf.shelf/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Shelf": S.OfObject, "Item": S.OfObject, "Holds": S.OfRelation})
        .spec([
            lambda _: _.Shelf.name("shelf.Shelf").ref(True).singleton("shelf.TheShelf").adjacencies({
                "items": lambda t: t.relation([_.Holds]).me("shelf")}),
            lambda _: _.Item.name("shelf.Item").ref(True).properties({"name": S.OfStr, "label": S.OfStr}).adjacencies([
                lambda t: t.name("shelf").relation([_.Holds]).me("item").owner(True)]),
            lambda _: _.Holds.name("shelf.Holds").links([lambda t: t.name("shelf"), lambda t: t.name("item")])
            .properties({"key": S.OfStr}).uniques([["shelf", "key"], ["item"]]),
        ])]))])


def patterns(txn):
    """shelf.Unlabelled, an item without a label, declaring the parameter case; shelf.Whole, the shelf; and
    shelf.Kinds, an object schema."""
    S = txn.Schemas
    return txn.uid("xf.patterns/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Unlabelled": S.OfPattern, "Whole": S.OfPattern, "Kinds": S.OfPattern})
        .spec([
            lambda _: _.Unlabelled.name("shelf.Unlabelled").symbols({"i": txn.shelf.Item})
            .spec([lambda f: f.i.label(None)])
            .parameters([txn.Parametrics.OfParameters(lambda ps: ps.parameters({
                "case": lambda d: d.parameter(lambda p: p.description("upper or lower").for_value([S.OfStr]))}))]),
            lambda _: _.Whole.name("shelf.Whole").symbols({"s": txn.shelf.Shelf}),
            lambda _: _.Kinds.name("shelf.Kinds").symbols({"o": S.OfObject}),
        ])]))])


def model(*names, with_patterns=True):
    """A store of the shelf's schemas and patterns, its shelf holding an item of each name, keyed by it."""
    store = B.Store()
    store.transaction(Parametrics.install)
    store.transaction(shelf)
    if with_patterns:
        store.transaction(patterns)
    store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"s": t.shelf.Shelf})]))]))
    for name in names:
        with store.transaction():
            store.shelf.Shelf(store.singleton("shelf.TheShelf"), lambda s, n=name: s.items({
                n: lambda e: e.item(lambda i: i.name(n))}))
    return store


def frames(store, pattern):
    """The frames of a pattern in a store, as the executor gives them back."""
    held = store.registered(pattern)
    return store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([held]))]))


def unlabelled(store, name):
    """The frame of shelf.Unlabelled binding the item `name`."""
    (frame,) = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([
        lambda p: p.symbols({"i": t.shelf.Item}).spec([lambda _: _.i.name(name).label(None)])]))]))
    return frame


def items(store):
    """Each item on the shelf: its key, name and label."""
    with store.transaction():
        return [(e.key.get(), e.item.name.get(), e.item.label.get()) for e in store.singleton("shelf.TheShelf").items]


def arguments_of(certificate):
    """The parameters' values a transform certificate records, read through the visitor protocol."""
    held = {}

    def each(entries):
        for name, value in entries:
            value.as_scalar(lambda s, name=name: held.__setitem__(name, s.get()) or s)
        return entries

    certificate.arguments.as_value(lambda a: a.as_container(lambda c: c.as_entries(each) or c) or a)
    return held


def steps(store, history="run"):
    """The number of transform certificates in a history."""
    with store.transaction():
        try:
            return len(store.singleton("Transforms.TheHistories").histories.at(history).history.steps)
        except KeyError:
            return 0


certs = model()  # the certificates' store: the shelf's schemas and patterns, no data, and the transforms
certs.transaction(Transforms.install)
for define in (
    lambda t: Transforms.define(t, "Label", "label", t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}, "falsified"),
    lambda t: Transforms.define(t, "Copy", "copy", t.shelf.Unlabelled, {"source": t.shelf.Unlabelled,
                                                                        "target": t.shelf.Whole}),
):
    certs.transaction(define)
procedures = Transforms.Procedures()


@procedures.register("label")
def label(source, sync, target, arguments):
    """Labels the sync's item with its name, in the case its argument gives."""
    item = sync.i
    name = str(item.name)
    Proxies.store_of(item).shelf.Item(item, lambda i: i.label(name.upper() if arguments["case"] == "upper" else name.lower()))


@procedures.register("copy")
def copy(source, sync, target, arguments):
    """Puts on the target's shelf an item named as the source's, in the case its argument gives."""
    name = str(source.i.name)
    name = name.upper() if arguments["case"] == "upper" else name.lower()
    Proxies.store_of(target.s).shelf.Shelf(target.s, lambda s: s.items({name: lambda e: e.item(lambda i: i.name(name))}))

<a id="XF-01"></a>
## XF-01 · A transform's anatomy: its name, prototype, frames' patterns, parameters, procedure and verification, defined as data

In [ ]:
Unlabelled, Whole = certs.registered("shelf.Unlabelled"), certs.registered("shelf.Whole")
with certs.transaction():
    entry = certs.singleton("Transforms.TheCatalog").transforms.at("Label")
    Label = entry.transform
    assert (str(entry.name), str(Label.procedure), str(Label.verify)) == ("Label", "label", "falsified")
    assert [e.pattern for e in Label.prototype] == [Unlabelled]
    assert {str(e.role): e.pattern for e in Label.frames} == {"sync": Unlabelled}
    Copy = certs.singleton("Transforms.TheCatalog").transforms.at("Copy").transform
    assert Copy.verify.get() is None and {str(e.role): e.pattern.name for e in Copy.frames} == {
        "source": "shelf.Unlabelled", "target": "shelf.Whole"}
    (held,) = B.OfPattern.Builder(Unlabelled, False, certs).parameters  # its parameters: its prototype's
    assert [str(e.name) for e in held.params.parameters] == ["case"]
assert Parametrics.validate(certs, Unlabelled) == []
snapshot = Plain.ToPlain(certs).Snapshot()
certs.transaction(Transforms.install)  # installed again, defined again: found, nothing written
certs.transaction(lambda t: Transforms.define(t, "Copy", "copy", t.shelf.Unlabelled, {
    "source": t.shelf.Unlabelled, "target": t.shelf.Whole}))
assert Plain.ToPlain(certs).Snapshot() == snapshot
assert certs.registered("Transforms.Transform").ref and certs.registered("Transforms.TransformCertificate").ref
for frames_, verify, message in [
    ({}, None, "a transform takes one to three frames"),
    ({"input": Unlabelled}, None, "each a role of ['source', 'sync', 'target']: got ['input']"),
    ({"sync": Unlabelled}, "always", "a transform verifies one of ['falsified', 'metric'], or nothing: got 'always'"),
]:
    with raises(ValueError, match=message):
        certs.transaction(lambda t: Transforms.define(t, "Odd", "label", Unlabelled, frames_, verify))

<a id="XF-02"></a>
## XF-02 · Procedures are registered by name, each once; an unknown name is refused before anything runs

In [ ]:
assert "label" in procedures and "ghost" not in procedures
with raises(ValueError, match="a procedure is registered as 'label' already"):
    procedures.register("label", label)
other = Transforms.Procedures()  # a registry is a value: none is global
assert "label" not in other and other.register("label", label) is label
certs.transaction(lambda t: Transforms.define(t, "Ghost", "ghost", t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}))
data = model("bolt")
with raises(LookupError, match="no procedure is registered as 'ghost'"):
    Transforms.apply(certs, "Ghost", {"case": "upper"}, "run", procedures, sync=Transforms.Bound(data, unlabelled(data, "bolt")))
assert steps(certs) == 0 and items(data) == [("bolt", "bolt", None)]

<a id="XF-03"></a>
## XF-03 · Frames are presented as `Visitors.OfPattern`: its pattern, as the frame's store holds it, and its symbols as members, each the visitor of the object it binds

In [ ]:
seen = []


@procedures.register("inspect")
def inspect(source, sync, target, arguments):
    """Records what it is given."""
    seen.append((repr(source), source.as_schema() is data.registered("shelf.Unlabelled"), str(source.i.name),
                 source.i.as_schema().name, sync, target, dict(arguments)))
    with raises(AttributeError, match="'nope' is not a symbol of the pattern 'shelf.Unlabelled'"):
        source.nope
    with raises(AttributeError):
        getattr(source, "_members_of")
    with raises(AttributeError, match="never assigned"):
        source.i = None
    with raises(TypeError):
        arguments["case"] = "lower"  # read-only: what the certificate records is what it was given


certs.transaction(lambda t: Transforms.define(t, "Inspect", "inspect", t.shelf.Unlabelled, {"source": t.shelf.Unlabelled}))
Transforms.apply(certs, "Inspect", {"case": "upper"}, "inspect", procedures, source=Transforms.Bound(data, unlabelled(data, "bolt")))
assert seen == [("<frame of shelf.Unlabelled>", True, "bolt", "shelf.Item", None, None, {"case": "upper"})]

kinds = []  # a frame binding a schema object holds it as itself, and so is it presented
procedures.register("kinds", lambda source, sync, target, arguments: kinds.append(source.o.name))
certs.transaction(lambda t: Transforms.define(t, "Kinds", "kinds", t.shelf.Kinds, {"source": t.shelf.Kinds}))
(item_kind,) = [frame for frame in frames(data, "shelf.Kinds") if frame["o"].name == "shelf.Item"]
Transforms.apply(certs, "Kinds", {}, "kinds", procedures, source=Transforms.Bound(data, item_kind))
assert kinds == ["shelf.Item"]

<a id="XF-04"></a>
## XF-04 · The source is read-only: what a procedure writes there is rolled back, the source undisturbed; the target is constructive, and the sync and the target may share a store

In [ ]:
@procedures.register("vandal")
def vandal(source, sync, target, arguments):
    """Writes the source, which is rolled back, and the target."""
    Proxies.store_of(source.i).shelf.Item(source.i, lambda i: i.label("defaced"))
    copy(source, sync, target, arguments)


certs.transaction(lambda t: Transforms.define(t, "Vandal", "vandal", t.shelf.Unlabelled, {
    "source": t.shelf.Unlabelled, "target": t.shelf.Whole}))
source, target = model("bolt", "nut"), model()
(whole,) = frames(target, "shelf.Whole")
Transforms.apply(certs, "Vandal", {"case": "upper"}, "copies", procedures,
                 source=Transforms.Bound(source, unlabelled(source, "bolt")), target=Transforms.Bound(target, whole))
assert items(source) == [("bolt", "bolt", None), ("nut", "nut", None)]  # undisturbed
assert items(target) == [("BOLT", "BOLT", None)]  # built

procedures.register("tag", lambda source, sync, target, arguments: (
    label(source, sync, target, arguments), copy(sync, None, target, {"case": "upper"})))
certs.transaction(lambda t: Transforms.define(t, "Tag", "tag", t.shelf.Unlabelled, {
    "sync": t.shelf.Unlabelled, "target": t.shelf.Whole}))
both = model("nut")  # one store, one transaction, for the sync and the target
(whole,) = frames(both, "shelf.Whole")
Transforms.apply(certs, "Tag", {"case": "lower"}, "copies", procedures,
                 sync=Transforms.Bound(both, unlabelled(both, "nut")), target=Transforms.Bound(both, whole))
assert items(both) == [("nut", "nut", "nut"), ("NUT", "NUT", None)]  # labelled in place, and copied

<a id="XF-05"></a>
## XF-05 · Parameters: every one the prototype declares is given a value, and no other; the transform certificate records them, and the history who decided them

In [ ]:
data = model("bolt", "nut")
for arguments, message in [({}, "missing ['case'], unknown []"), ({"case": "upper", "size": 3}, "missing [], unknown ['size']")]:
    with raises(ValueError, match=f"transform 'Label' has parameters ['case']: every one is given a value and no other is ({message})"):
        Transforms.apply(certs, "Label", arguments, "run", procedures, sync=Transforms.Bound(data, unlabelled(data, "bolt")))
with raises(ValueError, match="a step's parameters are decided by one of ['caller', 'policy', 'reused']: got 'chance'"):
    Transforms.apply(certs, "Label", {"case": "upper"}, "run", procedures, sync=Transforms.Bound(data, unlabelled(data, "bolt")),
                     by="chance")
first = Transforms.apply(certs, "Label", {"case": "upper"}, "run", procedures, sync=Transforms.Bound(data, unlabelled(data, "bolt")))
Transforms.apply(certs, "Label", {"case": "lower"}, "run", procedures, sync=Transforms.Bound(data, unlabelled(data, "nut")),
                 by="policy")
assert items(data) == [("bolt", "bolt", "BOLT"), ("nut", "nut", "nut")]
with certs.transaction():
    history = certs.singleton("Transforms.TheHistories").histories.at("run").history
    assert [(int(e.index), e.step.arguments.get()) for e in history.steps] == [(0, None), (1, None)]  # a container: not a scalar
    assert [arguments_of(e.step) for e in history.steps] == [{"case": "upper"}, {"case": "lower"}]
    assert [str(d.by) for d in history.decisions] == ["caller", "policy"]  # provenance: the history's, by step
    assert [d.step for d in history.decisions] == [e.step for e in history.steps]
assert Plain.ToPlain(certs)(first.as_schema(), first)["path"]["next"]["entry"]["next"]["entry"]["properties"] == {"index": 0}

<a id="XF-06"></a>
## XF-06 · Certificates are written and linked: the history's sequence, prev and next, a sub-certificate for each frame present holding its pattern instance, and the chain of each pattern instance; all of it data, written and read back as any

In [ ]:
source, target = model("bolt"), model()
(whole,) = frames(target, "shelf.Whole")
bolt = unlabelled(source, "bolt")
for case in ("upper", "lower"):  # the same pattern instances, twice
    Transforms.apply(certs, "Copy", {"case": case}, "chain", procedures,
                     source=Transforms.Bound(source, bolt), target=Transforms.Bound(target, whole))
assert items(target) == [("BOLT", "BOLT", None), ("bolt", "bolt", None)]
with certs.transaction():
    history = certs.singleton("Transforms.TheHistories").histories.at("chain").history
    one, two = (e.step for e in history.steps)
    assert [e.after for e in one.next] == [two] and [e.before for e in two.prev] == [one]
    assert list(one.prev) == [] and list(two.next) == []
    assert [e.transform for e in one.transform] == [certs.singleton("Transforms.TheCatalog").transforms.at("Copy").transform]
    assert [e.pattern for e in one.prototype] == [certs.registered("shelf.Unlabelled")]
    assert [str(e.role) for e in one.frames] == ["source", "target"]
    first_source, first_target = (e.frame for e in one.frames)
    second_source, second_target = (e.frame for e in two.frames)
    assert [e.pattern.name for e in first_target.pattern] == ["shelf.Whole"]
    assert list(first_source.preceding) == [] and [e.step for e in second_source.preceding] == [one]
    assert [e.step for e in second_target.preceding] == [one]
    held = []
    second_source.frame.as_object(lambda o: held.append(Plain.ToPlain(certs)(certs.References.OfFrame.as_schema(), o)) or o)
    (instance,) = held
expected = References.of(source, bolt)
assert instance == Plain.ToPlain(source)(expected.as_schema(), expected)  # the pattern instance: the frame, by reference

log = Plain.ToPlain(certs).Snapshot()  # the certificates are data: written and read back with no code of their own
back = B.Store()
Plain.FromPlain(back).Snapshot(log)
with back.transaction():
    one, two = (e.step for e in back.singleton("Transforms.TheHistories").histories.at("chain").history.steps)
    assert [arguments_of(one), arguments_of(two)] == [{"case": "upper"}, {"case": "lower"}]
    assert [e.before for e in two.prev] == [one] and [e.pattern.name for e in one.prototype] == ["shelf.Unlabelled"]

<a id="XF-07"></a>
## XF-07 · Verification by falsification: the prototype holds for the sync frame before, and no longer after; where it still holds, no certificate is written

In [ ]:
data = model("bolt")
procedures.register("idle", lambda source, sync, target, arguments: None)
certs.transaction(lambda t: Transforms.define(t, "Idle", "idle", t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}, "falsified"))
bolt = unlabelled(data, "bolt")
with raises(Transforms.VerificationError, match="transform 'Idle': its prototype still holds for the sync frame"):
    Transforms.apply(certs, "Idle", {"case": "upper"}, "falsify", procedures, sync=Transforms.Bound(data, bolt))
assert steps(certs, "falsify") == 0
Transforms.apply(certs, "Label", {"case": "upper"}, "falsify", procedures, sync=Transforms.Bound(data, bolt))
assert steps(certs, "falsify") == 1 and items(data) == [("bolt", "bolt", "BOLT")]
with raises(ValueError, match="its prototype does not hold for the sync frame, so nothing is there to falsify"):
    Transforms.apply(certs, "Label", {"case": "upper"}, "falsify", procedures, sync=Transforms.Bound(data, bolt))
certs.transaction(lambda t: Transforms.define(t, "Blind", "copy", t.shelf.Unlabelled, {
    "source": t.shelf.Unlabelled, "target": t.shelf.Whole}, "falsified"))
with raises(ValueError, match="transform 'Blind' verifies falsification: it takes a sync whose pattern is its prototype"):
    Transforms.apply(certs, "Blind", {"case": "upper"}, "falsify", procedures, source=Transforms.Bound(data, bolt),
                     target=Transforms.Bound(model(), {"s": bolt["i"]}))
assert steps(certs, "falsify") == 1

<a id="XF-08"></a>
## XF-08 · Verification by a metric: registered with the procedure, computed before and after it; where it does not decrease, nothing is written

In [ ]:
def unlabelled_count(source, sync, target, arguments):
    """How many items on the sync's shelf have no label."""
    return sum(1 for e in Proxies.store_of(sync.i).singleton("shelf.TheShelf").items if e.item.label.get() is None)


procedures.register("shrink", label, metric=unlabelled_count)
procedures.register("rename", lambda source, sync, target, arguments: Proxies.store_of(sync.i).shelf.Item(
    sync.i, lambda i: i.name("renamed")), metric=unlabelled_count)
for name, procedure in (("Shrink", "shrink"), ("Rename", "rename")):
    certs.transaction(lambda t: Transforms.define(t, name, procedure, t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}, "metric"))
data = model("bolt", "nut")
with raises(Transforms.VerificationError, match="transform 'Rename': its metric did not decrease (2 to 2)"):
    Transforms.apply(certs, "Rename", {"case": "upper"}, "metric", procedures, sync=Transforms.Bound(data, unlabelled(data, "bolt")))
assert items(data) == [("bolt", "bolt", None), ("nut", "nut", None)] and steps(certs, "metric") == 0  # rolled back
Transforms.apply(certs, "Shrink", {"case": "upper"}, "metric", procedures, sync=Transforms.Bound(data, unlabelled(data, "bolt")))
assert items(data) == [("bolt", "bolt", "BOLT"), ("nut", "nut", None)] and steps(certs, "metric") == 1
certs.transaction(lambda t: Transforms.define(t, "Measured", "copy", t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}, "metric"))
with raises(ValueError, match="transform 'Measured' verifies a metric: procedure 'copy' is registered with none"):
    Transforms.apply(certs, "Measured", {"case": "upper"}, "metric", procedures, sync=Transforms.Bound(data, unlabelled(data, "nut")))

<a id="XF-09"></a>
## XF-09 · Lifetimes: a procedure keeps nothing it was given; its frames and every visitor in them are refused once the application ends, and one that raises writes nothing

In [ ]:
kept = []
procedures.register("keep", lambda source, sync, target, arguments: kept.append((sync, sync.i)) or label(
    source, sync, target, arguments))
certs.transaction(lambda t: Transforms.define(t, "Keep", "keep", t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}))
data = model("bolt")
Transforms.apply(certs, "Keep", {"case": "lower"}, "keep", procedures, sync=Transforms.Bound(data, unlabelled(data, "bolt")))
((frame, item),) = kept
for use in (lambda: frame.i, lambda: frame.as_schema(), lambda: str(item.name)):
    with raises(ValueError, match="lives only within"):
        use()


def broken(source, sync, target, arguments):
    label(source, sync, target, arguments)
    raise RuntimeError("the procedure failed")


procedures.register("broken", broken)
certs.transaction(lambda t: Transforms.define(t, "Broken", "broken", t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}))
data = model("bolt")
with raises(RuntimeError, match="the procedure failed"):
    Transforms.apply(certs, "Broken", {"case": "upper"}, "broken", procedures, sync=Transforms.Bound(data, unlabelled(data, "bolt")))
assert items(data) == [("bolt", "bolt", None)] and steps(certs, "broken") == 0

<a id="XF-10"></a>
## XF-10 · What an application refuses: a transform not defined or ill-defined, frames that are not the transform's, stores shared where they must not be, and a frame its store cannot bind

In [ ]:
data, other = model("bolt"), model("nut")
bolt = unlabelled(data, "bolt")
(whole,) = frames(other, "shelf.Whole")
sync = Transforms.Bound(data, bolt)
with raises(LookupError, match="the store has no Transforms installed: store.transaction(Transforms.install)"):
    Transforms.apply(model(), "Label", {"case": "upper"}, "run", procedures, sync=sync)
with raises(LookupError, match="no transform named 'Nothing' is defined in the store"):
    Transforms.apply(certs, "Nothing", {}, "run", procedures, sync=sync)
certs.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"catalog": t.Transforms.Catalog, "transform": t.Transforms.Transform}).spec([
        lambda _: _.catalog.transforms({"Bare": lambda e: e.transform(_.transform)}),
        lambda _: _.transform.procedure("label")])]))]))
with raises(ValueError, match="transform 'Bare' has no prototype"):
    Transforms.apply(certs, "Bare", {}, "run", procedures, sync=sync)
certs.transaction(lambda t: Transforms.define(t, "Odd", "copy", t.shelf.Unlabelled, {"source": t.shelf.Whole, "target": t.shelf.Whole}))
certs.transaction(lambda t: Transforms.define(t, "Sometimes", "label", t.shelf.Unlabelled, {"sync": t.shelf.Unlabelled}))
with certs.transaction():
    certs.Transforms.Transform(certs.singleton("Transforms.TheCatalog").transforms.at("Sometimes").transform,
                               lambda x: x.verify("sometimes"))
source = Transforms.Bound(data, bolt)
for name, given, message in [
    ("Label", {"source": source}, "transform 'Label' takes ['sync']: given ['source']"),
    ("Odd", {"source": Transforms.Bound(other, whole), "target": Transforms.Bound(other, whole)},
     "transform 'Odd': its prototype is the pattern of its source frame"),
    ("Sometimes", {"sync": sync}, "transform 'Sometimes' verifies 'sometimes': one of ['falsified', 'metric'], or nothing"),
    ("Label", {"sync": Transforms.Bound(certs, bolt)}, "the sync's store is the certificates' store"),
    ("Copy", {"source": source, "target": Transforms.Bound(data, whole)}, "the target's store is the source's"),
    ("Label", {"sync": Transforms.Bound(data, Transactions.Frame(bolt, [Transactions.FrameError("$", "ValueError", "no")]))},
     "the sync frame is annotated with errors"),
    ("Label", {"sync": Transforms.Bound(data, {"j": bolt["i"]})}, "the sync frame binds ['j']: a frame of 'shelf.Unlabelled' binds ['i']"),
    ("Label", {"sync": Transforms.Bound(model("bolt", with_patterns=False), bolt)},
     "the sync's store holds no pattern 'shelf.Unlabelled': install it where its frames are bound"),
]:
    with raises(ValueError, LookupError, match=message):
        Transforms.apply(certs, name, {"case": "upper"}, "run", procedures, **given)
nowhere = Plain.FromPlain(data)(data.References.OfObject.as_schema(), {"path": {"singleton": "shelf.Nowhere", "next": {"target": {}}}})
item_kind = References.of(data, data.registered("shelf.Item"))
nut = unlabelled(other, "nut")["i"]  # the path of an item the data's store does not hold
nut = Plain.FromPlain(data)(data.References.OfObject.as_schema(), Plain.ToPlain(other)(nut.as_schema(), nut))
for frame, message in [({"i": nut}, "symbol 'i': the store holds nothing at its reference"),
                       ({"i": nowhere}, "symbol 'i': the store holds nothing at its reference"),
                       ({"i": item_kind}, "symbol 'i' is given a schema object's reference: a frame holds a schema object as itself")]:
    with raises(ValueError, LookupError, match=message):
        Transforms.apply(certs, "Copy", {"case": "upper"}, "run", procedures, source=Transforms.Bound(data, frame),
                         target=Transforms.Bound(other, whole))
assert items(data) == [("bolt", "bolt", None)] and items(other) == [("nut", "nut", None)] and steps(certs, "run") == 2